# Intro to Cognee - building a Personal Memory Assistant

## Why personal memory?

Personal attention is fragmented by default. A tiny obligation can arrive in a Slack mention, a family chat, an email thread, or a follow-up buried under a reply. The hard part is not just storing the messages. The hard part is noticing which ones still expect something from you.

A personal memory assistant should answer questions like:

- Who tagged me and still needs a reply?
- What did I already handle?
- What did I answer, but still need to finish?
- What is due by EOD, EOW, or a specific posted deadline?

This notebook builds that foundation with Cognee. We ingest a small personal corpus for Veljko: Slack-style channel exports and email-thread exports. Then we ask the graph for an EOD/EOW reminder brief.

The notebook follows a simple memory-assistant loop:

1. **Load data** - `remember()` ingests text and builds the graph in one call
2. **Recall** - ask natural-language questions over the personal memory
3. **Node sets** - tag Slack and email sources for scoping
4. **Custom graph model** - extract typed obligations and response state
5. **Add another source** - email joins the same graph as Slack
6. **Life-assistant brief** - ask for missed replies, handled items, and open follow-up


## The API (Cognee 1.x)

The modern surface is three async functions:

```python
await cognee.remember(text, dataset_name=...)   # 1. ingest + build the graph
await cognee.recall(query)                       # 2. ask questions
await cognee.improve(dataset=...)                # 3. refine the graph from feedback/use
```

`remember()` replaces the old two-step `add()` + `cognify()` flow for everyday use. It ingests the text, extracts graph structure, and runs the default self-improvement pass.

For a personal assistant, that means every normalized source - Slack, email, notes, calendar summaries - can become memory with the same call. The source matters less than the shape: clear speaker, timestamp, and message text.


## Setup

The setup cell below puts a placeholder `LLM_API_KEY` into `os.environ`. Replace `"your-api-key"` with a real key before running the notebook.

> Run once from the repo root: `uv pip install -e . jupyterlab ipykernel`


In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
os.environ.setdefault("LLM_API_KEY", "your-api-key")
os.environ.setdefault("CACHING", "true")

import cognee
from cognee.modules.users.methods import get_default_user
from cognee.modules.data.methods import get_authorized_existing_datasets
from cognee.context_global_variables import set_database_global_context_variables

# Works whether the kernel starts in the repo root or a notebook subdirectory.
ROOT = Path.cwd()
REPO = ROOT if (ROOT / "sample_data").exists() else ROOT.parent
DATA = REPO / "sample_data" / "personal_memory"
DATASET = "personal_memory"
TARGET_PERSON = "veljko@topoteretes.com"
TARGET_MENTION = "@veljko@topoteretes.com"

async def reset():
    """Wipe local Cognee state so the notebook can be re-run cleanly."""
    await cognee.prune.prune_data()
    await cognee.prune.prune_system(metadata=True)

def answer(results):
    """Pull a readable answer out of a recall() result list."""
    for r in results:
        for attr in ("text", "answer", "content"):
            value = getattr(r, attr, None)
            if value:
                return value
    return str(results[0]) if results else "<no answer>"

async def show_graph(filename, dataset, height=760):
    """Render a dataset-scoped graph to HTML, with open/download controls."""
    user = await get_default_user()
    datasets = await get_authorized_existing_datasets([dataset], "read", user)

    if not datasets:
        raise ValueError(f"Dataset not found or not readable: {dataset}")

    dataset_obj = datasets[0]
    owner_id = getattr(dataset_obj, "owner_id", None) or user.id

    out = ROOT / filename
    async with set_database_global_context_variables(dataset_obj.id, owner_id):
        await cognee.visualize_graph(str(out))

    from html import escape
    from IPython.display import HTML, IFrame, display

    href = escape(filename, quote=True)
    controls = (
        '<div style="display:flex;gap:8px;margin:8px 0 10px;align-items:center">'
        f'<a href="{href}" target="_blank" rel="noopener" '
        'style="padding:6px 10px;border:1px solid #ccc;border-radius:6px;text-decoration:none">'
        'Open full screen</a>'
        f'<a href="{href}" download '
        'style="padding:6px 10px;border:1px solid #ccc;border-radius:6px;text-decoration:none">'
        'Download HTML</a>'
        '</div>'
    )
    display(HTML(controls))
    return IFrame(src=filename, width="100%", height=height)


print("Data directory:", DATA)
print("Slack files:", [p.name for p in sorted(DATA.glob("slack_*.txt"))])
print("Email files:", [p.name for p in sorted(DATA.glob("email_*.txt"))])


2026-06-16T10:05:48.237189 [info     ] Deleted old log file: /Users/milenko/.cognee/logs/2026-06-16_09-27-30.log [cognee.shared.logging_utils]

2026-06-16T10:05:48.237572 [info     ] Log file created at: /Users/milenko/.cognee/logs/2026-06-16_12-05-48.log [cognee.shared.logging_utils] log_file=/Users/milenko/.cognee/logs/2026-06-16_12-05-48.log

2026-06-16T10:05:48.237867 [warning  ] Cognee 1.0 changes: New API — remember/recall/forget/improve (V1 add/cognify/search still work). Session memory enabled by default (CACHING=false to disable). Multi-user access control on by default (ENABLE_BACKEND_ACCESS_CONTROL=false to disable). Agents (@cognee.agent) auto-verified on registration. See https://docs.cognee.ai/ [cognee.shared.logging_utils]

2026-06-16T10:05:48.238080 [info     ] Logging initialized            [cognee.shared.logging_utils] cognee_version=1.1.2 database_path=/Users/milenko/cognee/cognee-companybrain-hackathon/.venv/lib/python3.13/site-packages/cognee/.cognee_system/databa

Data directory: /Users/milenko/cognee/cognee-companybrain-hackathon/sample_data/personal_memory
Slack files: ['slack_design.txt', 'slack_finance.txt', 'slack_health.txt', 'slack_product.txt', 'slack_social.txt']
Email files: ['email_landlord_lease_renewal.txt', 'email_parent_evening.txt', 'email_trip_documents.txt']


## 1. Load Slack messages with `remember()`

The Slack files use a lightweight export format: one channel transcript per file, with one line per message.

```text
# #channel-name: short thread title
[person@example.com, 2026-06-11T09:12] message text
```

In a real assistant, these would come from the Slack API. For the notebook, plain text keeps the data easy to inspect.


In [2]:
slack_files = sorted(DATA.glob("slack_*.txt"))
print(slack_files[0].read_text())


# #design-studio: Portfolio screenshots for Friday demo
[mina@topoteretes.com, 2026-06-11T09:12] @veljko@topoteretes.com can you upload three screenshots from the new memory graph flow before EOD 2026-06-11? I need them for the Friday demo deck.
[nikola@topoteretes.com, 2026-06-11T09:18] The deck is in Drive under Demo / June / Memory graph.
[mina@topoteretes.com, 2026-06-11T11:03] Reminder that the screenshots should show the inbox item, the extracted obligation, and the final reminder card.
[nikola@topoteretes.com, 2026-06-11T14:26] I updated slide 4, but I still do not have Veljko's screenshots.



In [3]:
await reset()

slack_files = sorted(DATA.glob("slack_*.txt"))
for f in slack_files:
    result = await cognee.remember(
        f.read_text(),
        dataset_name=DATASET,
        node_set=["source:slack", f"file:{f.stem}"],
    )
    print("remembered", f.name, "- status:", result.to_dict().get("status"))



2026-06-16T10:05:49.601514 [info     ] Deleted Ladybug database files at /Users/milenko/cognee/cognee-companybrain-hackathon/.venv/lib/python3.13/site-packages/cognee/.cognee_system/databases/db6bc087-57c9-422f-9e54-0288e379a1e0/fd8c0387-219f-5b82-b26f-fa1eeb08ee1b.lbug [cognee.shared.logging_utils]

2026-06-16T10:05:49.609284 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-16T10:05:49.663649 [info     ] Log file created at: /Users/milenko/.cognee/logs/2026-06-16_12-05-48.log [cognee.shared.logging_utils] log_file=/Users/milenko/.cognee/logs/2026-06-16_12-05-48.log

2026-06-16T10:05:49.663888 [warning  ] Cognee 1.0 changes: New API — remember/recall/forget/improve (V1 add/cognify/search still work). Session memory enabled by default (CACHING=false to disable). Multi-user access control on by default (ENABLE_BACKEND_ACCESS_CONTROL=false to disable). Agents (@cognee.agent) auto-verified on registration. See https://docs.cognee.ai/ [cognee.shared.l

remembered slack_design.txt - status: completed



2026-06-16T10:06:39.894505 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-16T10:06:39.895324 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-16T10:06:39.951884 [info     ] Pipeline run started: `3c566f60-5002-5038-9992-5870a1838b75` [run_tasks_with_telemetry()]

2026-06-16T10:06:39.958203 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2026-06-16T10:06:39.964545 [info     ] Async Generator task started: `extract_chunks_from_documents` [run_tasks_base]

2026-06-16T10:06:39.973664 [info     ] Coroutine task started: `extract_graph_and_summarize` [run_tasks_base]

2026-06-16T10:07:27.244678 [info     ] No close match found for 'person' in category 'classes' [OntologyAdapter]

2026-06-16T10:07:27.245232 [info     ] No close match found for 'tara@topoteretes.com' in category 'individuals' [OntologyAdapter]

2026-06-16T10:07:27.245535 [info     ] No close matc

remembered slack_finance.txt - status: completed



2026-06-16T10:07:29.623593 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-16T10:07:29.624329 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-16T10:07:29.681157 [info     ] Pipeline run started: `3c566f60-5002-5038-9992-5870a1838b75` [run_tasks_with_telemetry()]

2026-06-16T10:07:29.687395 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2026-06-16T10:07:29.693642 [info     ] Async Generator task started: `extract_chunks_from_documents` [run_tasks_base]

2026-06-16T10:07:29.708654 [info     ] Coroutine task started: `extract_graph_and_summarize` [run_tasks_base]

2026-06-16T10:08:14.047779 [info     ] No close match found for 'person' in category 'classes' [OntologyAdapter]

2026-06-16T10:08:14.048315 [info     ] No close match found for 'dragan@topoteretes.com' in category 'individuals' [OntologyAdapter]

2026-06-16T10:08:14.048636 [info     ] No close ma

remembered slack_health.txt - status: completed



2026-06-16T10:08:16.595175 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-16T10:08:16.595822 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-16T10:08:16.654184 [info     ] Pipeline run started: `3c566f60-5002-5038-9992-5870a1838b75` [run_tasks_with_telemetry()]

2026-06-16T10:08:16.660498 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2026-06-16T10:08:16.666794 [info     ] Async Generator task started: `extract_chunks_from_documents` [run_tasks_base]

2026-06-16T10:08:16.675713 [info     ] Coroutine task started: `extract_graph_and_summarize` [run_tasks_base]

2026-06-16T10:08:53.119752 [info     ] No close match found for 'person' in category 'classes' [OntologyAdapter]

2026-06-16T10:08:53.120308 [info     ] No close match found for 'ana@topoteretes.com' in category 'individuals' [OntologyAdapter]

2026-06-16T10:08:53.120692 [info     ] No close match

remembered slack_product.txt - status: completed



2026-06-16T10:08:55.416225 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-16T10:08:55.416898 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-16T10:08:55.477184 [info     ] Pipeline run started: `3c566f60-5002-5038-9992-5870a1838b75` [run_tasks_with_telemetry()]

2026-06-16T10:08:55.483493 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2026-06-16T10:08:55.489904 [info     ] Async Generator task started: `extract_chunks_from_documents` [run_tasks_base]

2026-06-16T10:08:55.498902 [info     ] Coroutine task started: `extract_graph_and_summarize` [run_tasks_base]

2026-06-16T10:09:50.199421 [info     ] No close match found for 'conversation' in category 'classes' [OntologyAdapter]

2026-06-16T10:09:50.199958 [info     ] No close match found for 'friends-weekend: saturday climbing plan' in category 'individuals' [OntologyAdapter]

2026-06-16T10:09:50.200227 

remembered slack_social.txt - status: completed


### Graph: Slack-only default extraction

This renders the first dataset while it still exists. The next section resets local state to keep the node-set demo isolated.

In [4]:
await show_graph("personal_memory_slack_default_graph.html", DATASET)



2026-06-16T10:09:52.616032 [info     ] Retrieved 77 nodes and 246 edges in 0.01 seconds [cognee.shared.logging_utils]

2026-06-16T10:09:52.620347 [info     ] Graph visualization saved as /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/personal_memory_slack_default_graph.html [cognee.shared.logging_utils]

2026-06-16T10:09:52.620575 [info     ] The HTML file has been stored at path: /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/personal_memory_slack_default_graph.html [cognee.shared.logging_utils]

2026-06-16T10:09:52.647824 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]


## 2. Recall - ask for missed replies

Now ask the graph like a personal assistant. The key distinction is not just "was Veljko mentioned?" but whether the thread shows a response or completion.


In [5]:
results = await cognee.recall(
    "Which Slack messages tagged Veljko and still need his response or action by EOD 2026-06-11? Separate missed replies from items where he replied but more work is needed.",
    datasets=[DATASET],
)
print(answer(results))



2026-06-16T10:09:52.668888 [info     ] query_router: routed=TEMPORAL score=3.0 query='Which Slack messages tagged Veljko and still need his response or action by EOD 2026-06-11? Separate missed replies from items where he replied but more work is needed.' scores={'TEMPORAL': 3.0} [query_router]

2026-06-16T10:10:12.007892 [info     ] No events identified based on timestamp filtering, performing retrieval using triplet search on events and entities. [cognee.shared.logging_utils]

2026-06-16T10:10:12.789611 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 0.78s [cognee.shared.logging_utils]

2026-06-16T10:10:12.790207 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-16T10:10:12.796687 [info     ] ID-filtered retrieval: 77 nodes and 246 edges in 0.01s [cognee.shared.logging_utils]

2026-06-16T10:10:12.798070 [info     ] Graph projection completed: 77 nodes, 246 edges in 0.00s [CogneeGraph]

2026-06-16T10:10:12.7989

Missed replies/actions by EOD 2026-06-11:
- #friends-weekend (Luka, 2026-06-11T07:55): "@veljko are you in for climbing on Saturday? Please answer by EOD 2026-06-11" — Veljko did not reply in the thread.
- #design-studio (Mina, 2026-06-11T09:12): "@veljko can you upload three screenshots from the new memory graph flow before EOD 2026-06-11?" — Nikola reports slide update but still missing Veljko’s screenshots (no upload/reply).

Replied but more work needed by EOD 2026-06-11:
- None in the provided data.


In [6]:
# Peek at the retrieved context behind the reminder answer.
ctx = await cognee.recall(
    "Veljko tagged EOD missed response open action",
    datasets=[DATASET],
    only_context=True,
    top_k=5,
)
for c in ctx:
    print("-", str(getattr(c, "content", c))[:300])



2026-06-16T10:10:26.053097 [info     ] query_router: no patterns matched, default=GRAPH_COMPLETION query='Veljko tagged EOD missed response open action' [query_router]

2026-06-16T10:10:26.922798 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 0.77s [cognee.shared.logging_utils]

2026-06-16T10:10:26.923438 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-16T10:10:26.931027 [info     ] ID-filtered retrieval: 77 nodes and 246 edges in 0.01s [cognee.shared.logging_utils]

2026-06-16T10:10:26.932544 [info     ] Graph projection completed: 77 nodes, 246 edges in 0.00s [CogneeGraph]

2026-06-16T10:10:26.933322 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 7, 'connection_count': 5}

2026-06-16T10:10:26.961167 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-16T10:10:27.000706 [info     ] recall: 1 results across sources=['graph']

- kind='graph_completion' search_type='GRAPH_COMPLETION' text="Nodes:\nNode: # #friends-weekend: Saturday climbing plan [luka@example.com, 2026-06-11T07:55]... [i, saturday, climbing]\n__node_content_start__\n# #friends-weekend: Saturday climbing plan\n[luka@example.com, 2026-06-11T07:55] @veljko@topo


## 3. Node sets - separate Slack from email

Node sets are tags attached at ingest time. Here we tag each source by channel/file and by broader source type, so the assistant can later answer from only Slack, only email, or the whole personal memory.

```python
await cognee.remember(text, dataset_name=DATASET, node_set=["source:slack"])
await cognee.remember(text, dataset_name=DATASET, node_set=["source:email"])
```


In [7]:
NS_DATASET = "personal_memory_node_sets"
await reset()

for f in sorted(DATA.glob("slack_*.txt")):
    await cognee.remember(f.read_text(), dataset_name=NS_DATASET, node_set=["source:slack", f"file:{f.stem}"])
for f in sorted(DATA.glob("email_*.txt")):
    await cognee.remember(f.read_text(), dataset_name=NS_DATASET, node_set=["source:email", f"file:{f.stem}"])

print("tagged Slack and email into separate node sets")



2026-06-16T10:10:27.157863 [info     ] Deleted Ladybug database files at /Users/milenko/cognee/cognee-companybrain-hackathon/.venv/lib/python3.13/site-packages/cognee/.cognee_system/databases/3a1e4a5c-979f-4557-a155-f6d1f25a77b5/2a6fff82-c745-5a9e-9cd8-006dbb6678ca.lbug [cognee.shared.logging_utils]

2026-06-16T10:10:27.165910 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-16T10:10:29.630877 [info     ] Database deleted successfully. [cognee.shared.logging_utils]

Deleting cache...             

✓ Cache deleted successfully! 

User 493533f7-a91e-4ca0-9658-7a2c8b997617 has registered.

2026-06-16T10:10:29.774298 [info     ] Pipeline run started: `81fc0d8c-8ffd-5ae2-b8f6-e35081406a85` [run_tasks_with_telemetry()]

2026-06-16T10:10:29.780494 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-06-16T10:10:29.786630 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-06-16T10:10:29.804165 [info 

tagged Slack and email into separate node sets


In [8]:
slack_only = await cognee.recall(
    "What Slack obligations are still open for Veljko?",
    datasets=[NS_DATASET],
    node_name=["source:slack"],
    scope=["graph"],
)
print(answer(slack_only))



2026-06-16T10:17:02.056386 [info     ] query_router: no patterns matched, default=GRAPH_COMPLETION query='What Slack obligations are still open for Veljko?' [query_router]

2026-06-16T10:17:02.972481 [info     ] Vector collection retrieval completed: Retrieved distances from 3 collections in 0.81s [cognee.shared.logging_utils]

2026-06-16T10:17:02.972961 [info     ] Retrieving graph filtered by node type and node name (NodeSet). [CogneeGraph]

2026-06-16T10:17:02.984674 [info     ] Graph projection completed: 49 nodes, 302 edges in 0.00s [CogneeGraph]

2026-06-16T10:17:02.985430 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 6, 'connection_count': 10}

2026-06-16T10:17:10.670650 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-16T10:17:10.724716 [info     ] recall: 1 results across sources=['graph'] (session=-) [recall]


Open Slack obligations for Veljko:
1) Upload three screenshots from the new memory-graph flow (showing the inbox item, the extracted-obligation UI element, and the final reminder card) for Mina’s Friday demo deck — due EOD 2026-06-11.
2) Reply to Luka about joining Saturday climbing (RSVP) — requested by EOD 2026-06-11.


### Graph: Slack and email node-set dataset

This dataset is tagged by source (`source:slack`, `source:email`) and by file. It shows how the same corpus can be organized for source scoping.

In [9]:
await show_graph("personal_memory_node_sets_graph.html", NS_DATASET)



2026-06-16T10:17:10.833430 [info     ] Retrieved 109 nodes and 391 edges in 0.01 seconds [cognee.shared.logging_utils]

2026-06-16T10:17:10.839061 [info     ] Graph visualization saved as /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/personal_memory_node_sets_graph.html [cognee.shared.logging_utils]

2026-06-16T10:17:10.839354 [info     ] The HTML file has been stored at path: /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/personal_memory_node_sets_graph.html [cognee.shared.logging_utils]

2026-06-16T10:17:10.867902 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]


## 4. Custom graph model - extract obligations

Default extraction can answer questions, but a life assistant benefits from a typed schema. We care about people, messages, and obligations:

- who asked
- who owns the next action
- what the deadline is
- whether the state is `missed_response`, `open_followup`, or `handled`

The custom graph model below defines those concepts directly as `Person`, `Message`, `Obligation`, and `MemoryThread`. The graph visualization in this section should therefore contain these newly defined model nodes, rather than only Cognee's default `Entity` / `EntityType` nodes.

The extraction prompt is intentionally strict: acknowledging a request is not the same as completing it.


In [10]:
from typing import Optional
from cognee.low_level import DataPoint

class Person(DataPoint):
    email: str
    name: str
    metadata: dict = {"index_fields": ["email", "name"], "identity_fields": ["email"]}

class Obligation(DataPoint):
    summary: str
    owner: Person
    requested_by: Person
    deadline: str
    status: str
    next_action: str
    evidence: str
    source: str
    metadata: dict = {"index_fields": ["summary", "deadline", "status", "next_action"]}

class Message(DataPoint):
    speaker: Person
    text: str
    sent_at: str
    mentions: Optional[list[Person]] = None
    obligations: Optional[list[Obligation]] = None
    metadata: dict = {"index_fields": ["text", "sent_at"]}

class MemoryThread(DataPoint):
    source: str
    title: str
    participants: list[Person]
    messages: list[Message]
    obligations: Optional[list[Obligation]] = None
    metadata: dict = {"index_fields": ["source", "title"]}


In [11]:
EXTRACTION_PROMPT = f"""Extract a personal memory graph from this Slack or email thread.

Target person: {TARGET_PERSON}

Use the provided MemoryThread graph model with Person, Message, and Obligation nodes.
Keep source evidence precise and short.

For every request, question, or deadline directed at the target person, create an Obligation.
Set status as one of:
- missed_response: the target person was asked or tagged and no later target-person reply appears.
- open_followup: the target person replied or acknowledged, but the text does not show completion, or a later message says the work is still missing.
- handled: the target person replied and the thread shows the request was completed or no more action is needed.

Deadline rules:
- Preserve explicit dates and times when present.
- Preserve EOD and EOW wording together with any stated weekday or date.
- Preserve explicit dates such as Thursday 2026-06-11 and Friday 2026-06-12.

Do not mark acknowledgement as handled unless completion is explicit.
Prefer exact source text over summaries."""

TYPED_DATASET = "personal_memory_typed"
await reset()

for f in sorted(DATA.glob("slack_*.txt")):
    await cognee.remember(
        f.read_text(),
        dataset_name=TYPED_DATASET,
        node_set=["source:slack", f"file:{f.stem}"],
        graph_model=MemoryThread,
        custom_prompt=EXTRACTION_PROMPT,
    )

print("typed Slack memory graph built")



2026-06-16T10:17:11.038374 [info     ] Deleted Ladybug database files at /Users/milenko/cognee/cognee-companybrain-hackathon/.venv/lib/python3.13/site-packages/cognee/.cognee_system/databases/493533f7-a91e-4ca0-9658-7a2c8b997617/d92c5c6b-fc49-537e-8f0b-211f4e3eb24a.lbug [cognee.shared.logging_utils]

2026-06-16T10:17:11.046379 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-16T10:17:13.516382 [info     ] Database deleted successfully. [cognee.shared.logging_utils]

Deleting cache...             

✓ Cache deleted successfully! 

User 6a6e87d0-f7a6-414f-badb-69701fc61b3e has registered.

2026-06-16T10:17:13.657542 [info     ] Pipeline run started: `1735cd44-6362-5471-a547-6278bbed6db9` [run_tasks_with_telemetry()]

2026-06-16T10:17:13.663662 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-06-16T10:17:13.669809 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-06-16T10:17:13.686561 [info 

typed Slack memory graph built


### Graph: Custom obligation model

This graph is built with the custom `MemoryThread` schema. Look for nodes shaped by the notebook's model definitions, especially `Person`, `Message`, `Obligation`, and `MemoryThread`, instead of the generic default extraction types.

In [12]:
await show_graph("personal_memory_custom_model_slack_graph.html", TYPED_DATASET)



2026-06-16T10:23:00.171891 [info     ] Retrieved 67 nodes and 127 edges in 0.01 seconds [cognee.shared.logging_utils]

2026-06-16T10:23:00.177521 [info     ] Graph visualization saved as /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/personal_memory_custom_model_slack_graph.html [cognee.shared.logging_utils]

2026-06-16T10:23:00.177893 [info     ] The HTML file has been stored at path: /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/personal_memory_custom_model_slack_graph.html [cognee.shared.logging_utils]

2026-06-16T10:23:00.207410 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]


In [13]:
typed_slack = await cognee.recall(
    "List Veljko's Slack obligations grouped by status: missed_response, open_followup, handled. Include deadline and source evidence.",
    datasets=[TYPED_DATASET],
)
print(answer(typed_slack))



2026-06-16T10:23:00.224449 [info     ] query_router: no patterns matched, default=GRAPH_COMPLETION query="List Veljko's Slack obligations grouped by status: missed_response, open_followup, handled. Include deadline and source evidence." [query_router]

2026-06-16T10:23:01.142942 [info     ] Vector collection retrieval completed: Retrieved distances from 24 collections in 0.81s [cognee.shared.logging_utils]

2026-06-16T10:23:01.143728 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-16T10:23:01.150233 [info     ] ID-filtered retrieval: 67 nodes and 127 edges in 0.01s [cognee.shared.logging_utils]

2026-06-16T10:23:01.151194 [info     ] Graph projection completed: 67 nodes, 127 edges in 0.00s [CogneeGraph]

2026-06-16T10:23:01.152302 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 15, 'connection_count': 10}

2026-06-16T10:23:23.384390 [info     ] Ladybug database closed successfully [cognee.shared.logging_ut

missed_response:
- Upload three screenshots (new memory graph flow) — deadline: EOD 2026-06-11. Evidence: Mina asked @veljko to upload three screenshots before EOD 2026-06-11 (2026-06-11T09:12); Mina reminder about what screenshots to show (2026-06-11T11:03); Nikola: “I updated slide 4, but I still do not have Veljko’s screenshots.” (2026-06-11T14:26).
- RSVP for Saturday climbing — deadline: EOD 2026-06-11. Evidence: Luka asked @veljko to answer by EOD 2026-06-11 so lanes can be reserved (2026-06-11T07:55). No Veljko reply present.
- Submit dentist insurance form to office (submit to their inbox) — deadline: before 2026-06-11 17:00 (office requirement); appointment 2026-06-12 09:30. Evidence: Dragan: dentist needs the form before appointment (2026-06-10T16:42); Veljko said he would fill it out that night (2026-06-10T17:05); Dragan: form must be in inbox before Thursday, 2026-06-11 at 17:00 or they may move the appointment (2026-06-11T09:14); Sofia checked family inbox and did not find

## 5. Add email threads to the same graph

Now add email as a second source. The format is different from Slack, but it is still plain text with people, timestamps, and messages. We use the same `remember()` call and the same typed schema.


In [14]:
email_files = sorted(DATA.glob("email_*.txt"))
print(email_files[0].read_text())


# Email thread: Lease renewal confirmation

From: Ana Petrovic <ana.petrovic@oakline-homes.example>
To: Veljko <veljko@topoteretes.com>
Date: 2026-06-09T08:18
Subject: Lease renewal paperwork due Friday

Hi Veljko,

We are preparing lease renewals for July. Could you confirm by Friday,
2026-06-12 at 12:00 whether you want to renew for another 12 months?

If yes, I will send the DocuSign packet. If you need different dates, please
reply with your preferred move-out or renewal term.

Ana

---

From: Veljko <veljko@topoteretes.com>
To: Ana Petrovic <ana.petrovic@oakline-homes.example>
Date: 2026-06-09T10:02
Subject: Re: Lease renewal paperwork due Friday

Hi Ana,

Yes, I want to renew for another 12 months. Please send the DocuSign packet.

Thanks,
Veljko

---

From: Ana Petrovic <ana.petrovic@oakline-homes.example>
To: Veljko <veljko@topoteretes.com>
Date: 2026-06-10T14:37
Subject: Re: Lease renewal paperwork due Friday

Thanks, Veljko. I sent the DocuSign packet just now. Please sign it

In [15]:
for f in sorted(DATA.glob("email_*.txt")):
    await cognee.remember(
        f.read_text(),
        dataset_name=TYPED_DATASET,
        node_set=["source:email", f"file:{f.stem}"],
        graph_model=MemoryThread,
        custom_prompt=EXTRACTION_PROMPT,
    )
    print("remembered", f.name)



2026-06-16T10:23:23.493603 [info     ] Pipeline run started: `1735cd44-6362-5471-a547-6278bbed6db9` [run_tasks_with_telemetry()]

2026-06-16T10:23:23.500140 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-06-16T10:23:23.506561 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-06-16T10:23:23.526602 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]

2026-06-16T10:23:23.533067 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]

2026-06-16T10:23:23.539417 [info     ] Pipeline run completed: `1735cd44-6362-5471-a547-6278bbed6db9` [run_tasks_with_telemetry()]

2026-06-16T10:23:23.682501 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-16T10:23:23.683320 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-16T10:23:23.745006 [info     ] Pipeline run started: `3bf26fd1-8faf-5984-bd7f-a2

remembered email_landlord_lease_renewal.txt



2026-06-16T10:24:36.550722 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-16T10:24:36.551407 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-16T10:24:36.614357 [info     ] Pipeline run started: `3bf26fd1-8faf-5984-bd7f-a2fbb6b21848` [run_tasks_with_telemetry()]

2026-06-16T10:24:36.620666 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2026-06-16T10:24:36.626888 [info     ] Async Generator task started: `extract_chunks_from_documents` [run_tasks_base]

2026-06-16T10:24:36.637835 [info     ] Coroutine task started: `extract_graph_and_summarize` [run_tasks_base]

2026-06-16T10:25:30.522263 [info     ] Coroutine task started: `add_data_points` [run_tasks_base]

2026-06-16T10:25:30.538814 [info     ] Completed graph extraction for DataPoint [cognee.shared.logging_utils] extra={'datapoint_id': '1a1ea6ca-7603-525b-bed4-ae30a099bb6c', 'nodes_extracted': 1, 'edg

remembered email_parent_evening.txt



2026-06-16T10:25:33.400046 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-16T10:25:33.400710 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-16T10:25:33.465856 [info     ] Pipeline run started: `3bf26fd1-8faf-5984-bd7f-a2fbb6b21848` [run_tasks_with_telemetry()]

2026-06-16T10:25:33.472164 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2026-06-16T10:25:33.478410 [info     ] Async Generator task started: `extract_chunks_from_documents` [run_tasks_base]

2026-06-16T10:25:33.487606 [info     ] Coroutine task started: `extract_graph_and_summarize` [run_tasks_base]

2026-06-16T10:26:14.764486 [info     ] Coroutine task started: `add_data_points` [run_tasks_base]

2026-06-16T10:26:14.782612 [info     ] Completed graph extraction for DataPoint [cognee.shared.logging_utils] extra={'datapoint_id': '1a1ea6ca-7603-525b-bed4-ae30a099bb6c', 'nodes_extracted': 1, 'edg

remembered email_trip_documents.txt


## 6. Life-assistant brief

This is the payoff: one typed obligation graph spanning Slack and email. Ask it for a reminder brief that a personal assistant could send at the end of the day or end of the week.


In [16]:
brief = await cognee.recall(
    "Create Veljko's reminder brief for Thursday 2026-06-11. Include: (1) missed replies due by EOD, (2) items he answered but still needs to finish, (3) items already handled and safe to ignore. Include source and deadline for each item.",
    datasets=[TYPED_DATASET],
)
print(answer(brief))



2026-06-16T10:26:17.429942 [info     ] query_router: routed=TEMPORAL score=3.0 query="Create Veljko's reminder brief for Thursday 2026-06-11. Include: (1) missed replies due by EOD, (2) items he answered but still needs to finish, (3) items already handled and safe to ignore. Include source and deadline for each item." scores={'TEMPORAL': 3.0} [query_router]

2026-06-16T10:26:22.764928 [info     ] No events identified based on timestamp filtering, performing retrieval using triplet search on events and entities. [cognee.shared.logging_utils]

2026-06-16T10:26:23.677088 [info     ] Vector collection retrieval completed: Retrieved distances from 30 collections in 0.91s [cognee.shared.logging_utils]

2026-06-16T10:26:23.677728 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-16T10:26:23.683828 [info     ] ID-filtered retrieval: 102 nodes and 192 edges in 0.01s [cognee.shared.logging_utils]

2026-06-16T10:26:23.685210 [info     ] Graph projection completed: 1

Reminder brief — Thursday 2026-06-11

1) Missed replies / actions due by EOD (need immediate response today)
- Upload three screenshots for Friday demo deck — deadline: EOD 2026-06-11. Source: Mina Slack request 2026-06-11T09:12; Mina reminder 2026-06-11T11:03; Nikola note still missing screenshots 2026-06-11T14:26.
- Reply re: Friday dinner (confirm whether you and Milica are coming) — deadline: EOD 2026-06-11. Source: Jelena email 2026-06-10 (asked for reply by EOD 2026-06-11); Milica CCed you asking you to decide 2026-06-11T08:46.
- RSVP for Saturday climbing (confirm attendance) — deadline: EOD 2026-06-11. Source: Luka Slack 2026-06-11T07:55 (please answer by EOD 2026-06-11).
- Submit dentist insurance form to the office (must be in their inbox) — deadline: before 2026-06-11 17:00 (office may move appointment). Source: Dragan Slack 2026-06-10T16:42; Dragan reminder 2026-06-11T09:14; Sofia checked inbox 2026-06-11T13:20 (form not found). Appointment: 2026-06-12 09:30.

2) Items you 

In [17]:
eow = await cognee.recall(
    "What should Veljko remember before EOW Friday 2026-06-12? Focus on unresolved obligations across Slack and email.",
    datasets=[TYPED_DATASET],
)
print(answer(eow))



2026-06-16T10:26:45.718225 [info     ] query_router: routed=TEMPORAL score=6.0 query='What should Veljko remember before EOW Friday 2026-06-12? Focus on unresolved obligations across Slack and email.' scores={'TEMPORAL': 6.0} [query_router]

2026-06-16T10:26:52.055437 [info     ] No events identified based on timestamp filtering, performing retrieval using triplet search on events and entities. [cognee.shared.logging_utils]

2026-06-16T10:26:52.928309 [info     ] Vector collection retrieval completed: Retrieved distances from 30 collections in 0.87s [cognee.shared.logging_utils]

2026-06-16T10:26:52.929092 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-16T10:26:52.935151 [info     ] ID-filtered retrieval: 102 nodes and 192 edges in 0.01s [cognee.shared.logging_utils]

2026-06-16T10:26:52.936514 [info     ] Graph projection completed: 102 nodes, 192 edges in 0.00s [CogneeGraph]

2026-06-16T10:26:52.937475 [info     ] Completed resolving edges to text [co

Unresolved items to handle before EOW Fri 2026-06-12 (deadlines and sources):

- Sign & return lease DocuSign packet — deadline: Fri 2026-06-12 12:00. Source: Email thread from Ana Petrovic (Ana sent DocuSign; asked to sign by 2026-06-12 12:00).

- Upload three screenshots for the Friday demo deck — deadline: EOD Thu 2026-06-11. Required shots: inbox item, extracted obligation, final reminder card. Source: #design-studio Slack (Mina 2026-06-11T09:12, reminder 11:03; Nikola 2026-06-11T14:26 reporting still missing).

- Reply to Jelena confirming whether you and Milica are coming to Friday dinner (final headcount) — deadline: EOD Thu 2026-06-11. Note: Milica can only attend if table is 20:30. Source: Email from Jelena Markovic (2026-06-10) and Milica’s reply CCing you (2026-06-11T08:46).

- RSVP for Saturday climbing (confirm attendance so Luka can reserve lanes) — deadline: EOD Thu 2026-06-11. Source: #friends-weekend Slack (Luka 2026-06-11T07:55).

- Billing-export dry run with product

## Start Local UI And Backend

Start Cognee local UI from the imported `cognee` library. The backend uses this notebook kernel environment, so it sees the same local Cognee settings and database paths as the tutorial.


In [19]:
import time
import urllib.request

import cognee

COGNEE_BACKEND_URL = "http://localhost:8000"
COGNEE_FRONTEND_URL = "http://localhost:3000"

cognee_ui_pids = []

def remember_cognee_ui_pid(pid_or_tuple):
    pid = pid_or_tuple[0] if isinstance(pid_or_tuple, tuple) else pid_or_tuple
    cognee_ui_pids.append(pid)
    print(f"Started Cognee UI process pid={pid}")

def url_ready(url: str, timeout_seconds: float = 1.5) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=timeout_seconds) as response:
            return 200 <= response.status < 500
    except Exception:
        return False

def wait_for_url(name: str, url: str, timeout_seconds: int = 90) -> None:
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        if url_ready(url):
            print(f"{name} ready: {url}")
            return
        time.sleep(1)
    print(f"{name} did not answer within {timeout_seconds}s: {url}")

cognee_frontend_process = cognee.start_ui(
    pid_callback=remember_cognee_ui_pid,
    port=3000,
    open_browser=False,
    auto_download=True,
    start_backend=True,
    backend_port=8000,
    start_mcp=False,
)

if cognee_frontend_process is None:
    raise RuntimeError("Cognee UI did not start. Check the cell output above for npm/backend errors.")

wait_for_url("Cognee backend", f"{COGNEE_BACKEND_URL}/health")
wait_for_url("Cognee frontend", COGNEE_FRONTEND_URL, timeout_seconds=120)

print(f"Open the local UI: {COGNEE_FRONTEND_URL}")



2026-06-16T10:27:09.304246 [info     ] Starting cognee UI...          [cognee.shared.logging_utils]

2026-06-16T10:27:09.304570 [info     ] Checking port availability...  [cognee.shared.logging_utils]

2026-06-16T10:27:09.305905 [info     ] ✓ All required ports are available [cognee.shared.logging_utils]

2026-06-16T10:27:09.306118 [info     ] Starting cognee backend API server... [cognee.shared.logging_utils]
/Users/milenko/.local/share/uv/python/cpython-3.13.12-macos-aarch64-none/lib/python3.13/subprocess.py:1921: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  self.pid = _fork_exec(


Started Cognee UI process pid=81371
[BACKEND] 2026-06-16T10:27:09.468898 [info     ] Log file created at: /Users/milenko/.cognee/logs/2026-06-16_12-05-48.log [cognee.shared.logging_utils] log_file=/Users/milenko/.cognee/logs/2026-06-16_12-05-48.log
[BACKEND] 2026-06-16T10:27:09.469015 [warning  ] Cognee 1.0 changes: New API — remember/recall/forget/improve (V1 add/cognify/search still work). Session memory enabled by default (CACHING=false to disable). Multi-user access control on by default (ENABLE_BACKEND_ACCESS_CONTROL=false to disable). Agents (@cognee.agent) auto-verified on registration. See https://docs.cognee.ai/ [cognee.shared.logging_utils]
[BACKEND] 2026-06-16T10:27:09.469067 [info     ] Logging initialized            [cognee.shared.logging_utils] cognee_version=1.1.2 database_path=/Users/milenko/cognee/cognee-companybrain-hackathon/.venv/lib/python3.13/site-packages/cognee/.cognee_system/databases os_info='Darwin 25.5.0 (Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT


2026-06-16T10:27:11.320938 [info     ] ✓ Backend API started at http://localhost:8000 [cognee.shared.logging_utils]


[BACKEND] /Users/milenko/cognee/cognee-companybrain-hackathon/.venv/lib/python3.13/site-packages/cognee/api/v1/update/routers/get_update_router.py:42: FastAPIDeprecationWarning: `example` has been deprecated, please use `examples` instead
[BACKEND]   node_set: Optional[List[str]] = Form(default=[""], example=[""]),
[BACKEND] /Users/milenko/cognee/cognee-companybrain-hackathon/.venv/lib/python3.13/site-packages/cognee/api/v1/remember/routers/get_remember_router.py:36: FastAPIDeprecationWarning: `example` has been deprecated, please use `examples` instead
[BACKEND]   node_set: Optional[List[str]] = Form(default=[""], example=[""]),
[BACKEND] INFO:     Started server process [81371]
[BACKEND] INFO:     Waiting for application startup.



2026-06-16T10:27:11.404648 [info     ] Starting frontend server at http://localhost:3000 [cognee.shared.logging_utils]

2026-06-16T10:27:11.405040 [info     ] This may take a moment to compile and start... [cognee.shared.logging_utils]
/Users/milenko/.local/share/uv/python/cpython-3.13.12-macos-aarch64-none/lib/python3.13/subprocess.py:1921: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  self.pid = _fork_exec(


Started Cognee UI process pid=81375
[FRONTEND] > cognee-frontend@1.0.0 dev
[FRONTEND] > next dev --turbopack
[FRONTEND] ▲ Next.js 16.1.6 (Turbopack)
[FRONTEND] - Local:         http://localhost:3000
[FRONTEND] - Network:       http://192.168.1.95:3000
[FRONTEND] ✓ Starting...
[FRONTEND] ⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
[FRONTEND] ✓ Ready in 416ms
[FRONTEND]  GET /socket.io?EIO=4&transport=polling&t=PxF-8Xm 404 in 244ms (compile: 154ms, proxy.ts: 22ms, render: 68ms)
[FRONTEND]  GET /socket.io?EIO=4&transport=polling&t=PxF-8gD 404 in 18ms (compile: 3ms, proxy.ts: 2ms, render: 13ms)
[BACKEND] Migration completed successfully.
[BACKEND] 2026-06-16T10:27:13.095613 [info     ] Log file created at: /Users/milenko/.cognee/logs/2026-06-16_12-05-48.log [cognee.shared.logging_utils] log_file=/Users/milenko/.cognee/logs/2026-06-16_12-05-48.log
[BACKEND] 2026-06-16T10:27:13.095694 [warning 


2026-06-16T10:27:14.420195 [info     ] ✓ Cognee UI is starting up...  [cognee.shared.logging_utils]

2026-06-16T10:27:14.421038 [info     ] ✓ Open your browser to: http://localhost:3000 [cognee.shared.logging_utils]

2026-06-16T10:27:14.421489 [info     ] ✓ The UI will be available once Next.js finishes compiling [cognee.shared.logging_utils]


[BACKEND] INFO:     ::1:51676 - "GET /health HTTP/1.1" 200 OK
Cognee backend ready: http://localhost:8000/health
Cognee frontend ready: http://localhost:3000
Open the local UI: http://localhost:3000
[FRONTEND]  GET / 200 in 68ms (compile: 30ms, proxy.ts: 1353µs, render: 36ms)


## Terminate Local UI And Backend

Run the next cell when you are done.


In [20]:
# import os
# import signal
# import time

# def process_is_running(pid: int) -> bool:
#     try:
#         os.kill(pid, 0)
#         return True
#     except OSError:
#         return False

# def stop_process_group(pid: int, timeout_seconds: int = 15) -> None:
#     if not process_is_running(pid):
#         print(f"Process pid={pid} is already stopped.")
#         return

#     print(f"Stopping Cognee UI process group pid={pid}")
#     try:
#         os.killpg(pid, signal.SIGTERM)
#     except ProcessLookupError:
#         print(f"Process group pid={pid} is already stopped.")
#         return

#     deadline = time.time() + timeout_seconds
#     while time.time() < deadline:
#         if not process_is_running(pid):
#             print(f"Stopped pid={pid}.")
#             return
#         time.sleep(0.5)

#     print(f"pid={pid} did not stop after SIGTERM; sending SIGKILL.")
#     try:
#         os.killpg(pid, signal.SIGKILL)
#     except ProcessLookupError:
#         pass

# for pid in reversed(globals().get("cognee_ui_pids", [])):
#     stop_process_group(pid)

# cognee_ui_pids = []
# cognee_frontend_process = None
